In [1]:
import os

# Connection settings come from the environment (see .env.example at the repo root).
PG = dict(
    host=os.environ.get("POSTGRES_HOST", "localhost"),
    port=os.environ.get("POSTGRES_PORT", "5432"),
    dbname=os.environ.get("ECOMMERCE_DB", "ecommerce"),
    user=os.environ.get("POSTGRES_USER", "postgres"),
    password=os.environ.get("POSTGRES_PASSWORD"),
)

import psycopg2
import pandas as pd

# Connect to PostgreSQL
conn = psycopg2.connect(**PG)

print("Connected successfully!")

Connected successfully!


In [2]:
# =========================================
# INSIGHT 1
# Average Purchase Amount by
# Age Group, Gender, Income Level
# =========================================
query1 = """
SELECT
    CASE
        WHEN age BETWEEN 18 AND 25 THEN '18-25'
        WHEN age BETWEEN 26 AND 35 THEN '26-35'
        WHEN age BETWEEN 36 AND 45 THEN '36-45'
        ELSE '46+'
    END AS age_group,
    gender,
    income_level,
    COUNT(*) AS total_customers,
    ROUND(AVG(purchase_amount), 2) AS avg_purchase_amount,
    ROUND(SUM(purchase_amount), 2) AS total_revenue
FROM public.ecommerce_cleaned
GROUP BY age_group, gender, income_level
ORDER BY total_revenue DESC;
"""

df1 = pd.read_sql(query1, conn)

print("\n=== INSIGHT 1 ===")
print(df1)



=== INSIGHT 1 ===
   age_group  gender income_level  total_customers  avg_purchase_amount  \
0      26-35  Female         High               80               278.19   
1      36-45    Male         High               71               284.80   
2      36-45  Female       Middle               68               297.00   
3      36-45    Male       Middle               73               269.31   
4      26-35    Male         High               70               269.06   
5      26-35    Male       Middle               74               250.69   
6      36-45  Female         High               75               247.08   
7      26-35  Female       Middle               57               320.10   
8      18-25  Female         High               53               305.92   
9      18-25  Female       Middle               53               273.38   
10     18-25    Male         High               47               266.61   
11     18-25    Male       Middle               42               270.22   
12    

<tmp>\ipykernel_21616\1773766096.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query1, conn)


In [3]:
# =========================================
# INSIGHT 2
# Purchase Frequency by Discount Sensitivity
# =========================================

query2 = """
SELECT
    discount_sensitivity,
    COUNT(*) AS total_customers,
    ROUND(AVG(frequency_of_purchase), 2) AS avg_purchase_frequency,
    ROUND(AVG(purchase_amount), 2) AS avg_purchase_amount
FROM public.ecommerce_cleaned
GROUP BY discount_sensitivity
ORDER BY avg_purchase_frequency DESC;
"""

df2 = pd.read_sql(query2, conn)

print("\n=== INSIGHT 2 ===")
print(df2)


=== INSIGHT 2 ===
  discount_sensitivity  total_customers  avg_purchase_frequency  \
0   Somewhat Sensitive              319                    7.09   
1        Not Sensitive              331                    6.93   
2       Very Sensitive              350                    6.83   

   avg_purchase_amount  
0               282.14  
1               267.48  
2               275.79  


<tmp>\ipykernel_21616\3300312223.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(query2, conn)


In [4]:
# =========================================
# INSIGHT 3
# Purchase Intent vs Engagement with Ads
# =========================================

query3 = """
SELECT
    engagement_with_ads,
    purchase_intent,
    COUNT(*) AS total_customers,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (PARTITION BY engagement_with_ads),
        2
    ) AS percentage_within_engagement
FROM public.ecommerce_cleaned
GROUP BY engagement_with_ads, purchase_intent
ORDER BY engagement_with_ads, total_customers DESC;
"""

df3 = pd.read_sql(query3, conn)

print("\n=== INSIGHT 3 ===")
print(df3)



=== INSIGHT 3 ===
   engagement_with_ads purchase_intent  total_customers  \
0                 High      Need-based               82   
1                 High         Planned               68   
2                 High     Wants-based               63   
3                 High       Impulsive               57   
4                  Low      Need-based               66   
5                  Low       Impulsive               56   
6                  Low         Planned               54   
7                  Low     Wants-based               54   
8               Medium     Wants-based               73   
9               Medium         Planned               60   
10              Medium       Impulsive               58   
11              Medium      Need-based               53   
12             Unknown       Impulsive               77   
13             Unknown         Planned               65   
14             Unknown     Wants-based               59   
15             Unknown      Need-base

<tmp>\ipykernel_21616\913107759.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql(query3, conn)


In [5]:
# =========================================
# INSIGHT 4
# Most Used Payment Method
# =========================================

query4 = """
SELECT
    payment_method,
    COUNT(*) AS total_transactions,
    ROUND(AVG(purchase_amount), 2) AS avg_transaction_value,
    ROUND(SUM(purchase_amount), 2) AS total_revenue
FROM public.ecommerce_cleaned
GROUP BY payment_method
ORDER BY total_transactions DESC;
"""

df4 = pd.read_sql(query4, conn)

print("\n=== INSIGHT 4 ===")
print(df4)


=== INSIGHT 4 ===
  payment_method  total_transactions  avg_transaction_value  total_revenue
0         PayPal                 219                 278.38       60964.33
1          Other                 204                 283.32       57797.50
2     Debit Card                 196                 273.22       53552.04
3    Credit Card                 194                 271.53       52677.54
4           Cash                 187                 267.77       50072.47


<tmp>\ipykernel_21616\3498469312.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df4 = pd.read_sql(query4, conn)


In [6]:
# =========================================
# INSIGHT 5
# Best Performing Purchase Channel
# =========================================

query5 = """
SELECT
    purchase_channel,
    COUNT(*) AS total_transactions,
    ROUND(AVG(purchase_amount), 2) AS avg_spending,
    ROUND(AVG(frequency_of_purchase), 2) AS avg_frequency,
    ROUND(SUM(purchase_amount), 2) AS total_revenue
FROM public.ecommerce_cleaned
GROUP BY purchase_channel
ORDER BY total_revenue DESC;
"""

df5 = pd.read_sql(query5, conn)

print("\n=== INSIGHT 5 ===")
print(df5)


=== INSIGHT 5 ===
  purchase_channel  total_transactions  avg_spending  avg_frequency  \
0            Mixed                 340        279.90           6.81   
1           Online                 334        274.26           7.03   
2         In-Store                 326        270.84           7.00   

   total_revenue  
0       95164.36  
1       91604.23  
2       88295.29  


<tmp>\ipykernel_21616\3024706151.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df5 = pd.read_sql(query5, conn)


In [7]:
# =========================================
# INSIGHT 6
# Most Purchased Categories by Demographic
# =========================================

query6 = """
SELECT
    gender,
    income_level,
    purchase_category,
    COUNT(*) AS total_purchases,
    ROUND(SUM(purchase_amount), 2) AS total_spent
FROM public.ecommerce_cleaned
GROUP BY gender, income_level, purchase_category
ORDER BY gender, income_level, total_purchases DESC;
"""

df6 = pd.read_sql(query6, conn)

print("\n=== INSIGHT 6 ===")
print(df6)



=== INSIGHT 6 ===
     gender income_level       purchase_category  total_purchases  total_spent
0    Female         High             Electronics               18      4238.43
1    Female         High   Jewelry & Accessories               17      4831.31
2    Female         High        Food & Beverages               16      5247.59
3    Female         High            Toys & Games               13      4115.10
4    Female         High      Mobile Accessories               12      3634.84
..      ...          ...                     ...              ...          ...
135   Other       Middle       Sports & Outdoors                1       480.52
136   Other       Middle  Beauty & Personal Care                1       108.50
137   Other       Middle            Toys & Games                1       207.18
138   Other       Middle   Jewelry & Accessories                1       489.66
139   Other       Middle               Groceries                1       491.92

[140 rows x 5 columns]


<tmp>\ipykernel_21616\2887914573.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df6 = pd.read_sql(query6, conn)


In [8]:
# =========================================
# INSIGHT 7
# Loyal Customers Favorite Categories
# =========================================

query7 = """
SELECT
    purchase_category,
    COUNT(*) AS total_purchases,
    ROUND(AVG(purchase_amount), 2) AS avg_purchase_amount,
    ROUND(SUM(purchase_amount), 2) AS total_revenue
FROM public.ecommerce_cleaned
WHERE loyalty_member = TRUE
GROUP BY purchase_category
ORDER BY total_purchases DESC;
"""

df7 = pd.read_sql(query7, conn)

print("\n=== INSIGHT 7 ===")
print(df7)


=== INSIGHT 7 ===
            purchase_category  total_purchases  avg_purchase_amount  \
0                 Electronics               39               244.58   
1       Jewelry & Accessories               28               283.30   
2             Home Appliances               27               254.26   
3          Health Supplements               26               258.13   
4                Toys & Games               26               280.34   
5                 Animal Feed               23               228.91   
6        Gardening & Outdoors               23               273.52   
7                   Packages)               22               297.93   
8                   Furniture               21               238.71   
9           Sports & Outdoors               20               279.31   
10               Luxury Goods               20               254.58   
11              Arts & Crafts               20               184.89   
12                Health Care               20            

<tmp>\ipykernel_21616\921000668.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df7 = pd.read_sql(query7, conn)


In [9]:
# =========================================
# INSIGHT 8
# Non-Loyal Customers Favorite Categories
# =========================================

query8 = """
SELECT
    purchase_category,
    COUNT(*) AS total_purchases,
    ROUND(AVG(purchase_amount), 2) AS avg_purchase_amount,
    ROUND(SUM(purchase_amount), 2) AS total_revenue
FROM public.ecommerce_cleaned
WHERE loyalty_member = FALSE
GROUP BY purchase_category
ORDER BY total_purchases DESC;
"""

df8 = pd.read_sql(query8, conn)

print("\n=== INSIGHT 8 ===")
print(df8)


=== INSIGHT 8 ===
            purchase_category  total_purchases  avg_purchase_amount  \
0           Sports & Outdoors               31               291.11   
1            Food & Beverages               27               303.28   
2             Software & Apps               27               331.34   
3                   Groceries               26               279.58   
4               Baby Products               23               261.22   
5             Home Appliances               23               275.08   
6                      Hotels               22               277.00   
7          Mobile Accessories               22               292.57   
8       Jewelry & Accessories               22               327.59   
9   Travel & Leisure (Flights               21               289.33   
10                  Packages)               21               294.12   
11                Animal Feed               21               295.35   
12                Health Care               21            

<tmp>\ipykernel_21616\3054745355.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df8 = pd.read_sql(query8, conn)


In [10]:
# =========================================
# INSIGHT 9
# Recommendation by Age Group
# =========================================

query9 = """
SELECT
    CASE
        WHEN age BETWEEN 18 AND 25 THEN '18-25'
        WHEN age BETWEEN 26 AND 35 THEN '26-35'
        WHEN age BETWEEN 36 AND 45 THEN '36-45'
        ELSE '46+'
    END AS age_group,
    purchase_category,
    COUNT(*) AS total_purchases,
    ROUND(SUM(purchase_amount), 2) AS total_spent
FROM public.ecommerce_cleaned
GROUP BY age_group, purchase_category
ORDER BY age_group, total_purchases DESC;
"""

df9 = pd.read_sql(query9, conn)

print("\n=== INSIGHT 9 ===")
print(df9)


=== INSIGHT 9 ===
   age_group   purchase_category  total_purchases  total_spent
0      18-25   Sports & Outdoors               17      4938.47
1      18-25  Mobile Accessories               16      4313.63
2      18-25        Toys & Games               14      3243.68
3      18-25     Office Supplies               14      3889.60
4      18-25     Software & Apps               14      4624.71
..       ...                 ...              ...          ...
91       46+              Hotels                4       719.59
92       46+               Books                4      1211.54
93       46+         Health Care                4      1213.59
94       46+        Luxury Goods                3       917.73
95       46+            Clothing                3       810.35

[96 rows x 4 columns]


<tmp>\ipykernel_21616\1947584581.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df9 = pd.read_sql(query9, conn)


In [11]:
# =========================================
# INSIGHT 10
# Top High-Spending Loyal Customers
# =========================================

query10 = """
SELECT
    customer_id,
    gender,
    income_level,
    COUNT(*) AS total_transactions,
    ROUND(SUM(purchase_amount), 2) AS total_spent,
    ROUND(AVG(purchase_amount), 2) AS avg_transaction_value
FROM public.ecommerce_cleaned
WHERE loyalty_member = TRUE
GROUP BY customer_id, gender, income_level
ORDER BY total_spent DESC
LIMIT 20;
"""

df10 = pd.read_sql(query10, conn)

print("\n=== INSIGHT 10 ===")
print(df10)


=== INSIGHT 10 ===
    customer_id  gender income_level  total_transactions  total_spent  \
0   60-470-3563    Male       Middle                   1       498.33   
1   86-257-9581    Male       Middle                   1       497.76   
2   72-590-6161  Female         High                   1       495.80   
3   59-261-4453    Male         High                   1       494.81   
4   37-468-8133    Male       Middle                   1       492.51   
5   38-940-4942  Female       Middle                   1       490.41   
6   56-810-2066   Other       Middle                   1       490.34   
7   29-625-7057  Female       Middle                   1       489.05   
8   10-870-3357    Male         High                   1       487.96   
9   34-788-8387  Female       Middle                   1       485.76   
10  43-616-8365  Female         High                   1       483.38   
11  76-620-2994    Male       Middle                   1       482.50   
12  03-023-9932  Female       M

<tmp>\ipykernel_21616\1179375973.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df10 = pd.read_sql(query10, conn)


In [12]:
# =========================================
# CLOSE CONNECTION
# =========================================

conn.close()

print("\nDatabase connection closed.")


Database connection closed.
